# 第2节:自然语言推理

本节介绍自然语言推理(Natural Language Inference, NLI)任务,包括SNLI数据集和基于注意力机制的可分解注意力模型。

## 学习目标

1. 理解自然语言推理任务的定义和应用
2. 掌握SNLI数据集的结构和处理方法
3. 理解可分解注意力模型的三个步骤:注意、比较、聚合
4. 实现基于注意力机制的NLI模型
5. 了解如何使用BERT进行NLI任务

## 2.1 自然语言推理简介

### 什么是自然语言推理?

在情感分析中,我们讨论了将单个文本序列分类到预定义的类别中。然而,当需要决定一个句子是否可以从另一个句子推断出来,或者需要通过识别语义等价的句子来消除句子间冗余时,知道如何对一个文本序列进行分类是不够的。相反,我们需要能够**对成对的文本序列进行推断**。

**自然语言推理**(natural language inference)主要研究**假设**(hypothesis)是否可以从**前提**(premise)中推断出来,其中两者都是文本序列。

换言之,自然语言推理决定了一对文本序列之间的逻辑关系。

### 三种关系类型

这类关系通常分为三种类型:

1. **蕴涵**(entailment):假设可以从前提中推断出来
2. **矛盾**(contradiction):假设的否定可以从前提中推断出来  
3. **中性**(neutral):所有其他情况

### 示例

#### 蕴涵示例

下面的一个文本对将被贴上"蕴涵"的标签,因为假设中的"表白"可以从前提中的"拥抱"中推断出来。

> **前提**:两个女人拥抱在一起。
>
> **假设**:两个女人在示爱。

#### 矛盾示例

下面是一个"矛盾"的例子,因为"运行编码示例"表示"不睡觉",而不是"睡觉"。

> **前提**:一名男子正在运行Dive Into Deep Learning的编码示例。
>
> **假设**:该男子正在睡觉。

#### 中性示例

第三个例子显示了一种"中性"关系,因为"正在为我们表演"这一事实无法推断出"出名"或"不出名"。

> **前提**:音乐家们正在为我们表演。
>
> **假设**:音乐家很有名。

### 应用场景

自然语言推理一直是理解自然语言的中心话题。它有着广泛的应用:

- 信息检索
- 开放领域的问答
- 文本摘要
- 对话系统

In [ ]:
import os
import re
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

## 2.2 斯坦福自然语言推理(SNLI)数据集

### 数据集简介

**斯坦福自然语言推理语料库**(Stanford Natural Language Inference, SNLI)是由500000多个带标签的英语句子对组成的集合。我们在路径`../data/snli_1.0`中下载并存储提取的SNLI数据集。

In [ ]:
#@save
d2l.DATA_HUB['SNLI'] = (
    'https://nlp.stanford.edu/projects/snli/snli_1.0.zip',
    '9fcde07509c7e87ec61c640c1b2753d9041758e4')

data_dir = d2l.download_extract('SNLI')

### 读取数据集

原始的SNLI数据集包含的信息比我们在实验中真正需要的信息丰富得多。因此,我们定义函数`read_snli`以仅提取数据集的一部分,然后返回前提、假设及其标签的列表。

In [ ]:
#@save
def read_snli(data_dir, is_train):
    """将SNLI数据集解析为前提、假设和标签"""
    def extract_text(s):
        # 删除我们不会使用的信息
        s = re.sub('\\(', '', s)
        s = re.sub('\\)', '', s)
        # 用一个空格替换两个或多个连续的空格
        s = re.sub('\\s{2,}', ' ', s)
        return s.strip()
    label_set = {'entailment': 0, 'contradiction': 1, 'neutral': 2}
    file_name = os.path.join(data_dir, 'snli_1.0_train.txt'
                             if is_train else 'snli_1.0_test.txt')
    with open(file_name, 'r') as f:
        rows = [row.split('\t') for row in f.readlines()[1:]]
    premises = [extract_text(row[1]) for row in rows if row[0] in label_set]
    hypotheses = [extract_text(row[2]) for row in rows if row[0] \
                in label_set]
    labels = [label_set[row[0]] for row in rows if row[0] in label_set]
    return premises, hypotheses, labels

现在让我们打印前3对前提和假设,以及它们的标签("0""1"和"2"分别对应于"蕴涵""矛盾"和"中性")。

In [ ]:
train_data = read_snli(data_dir, is_train=True)
for x0, x1, y in zip(train_data[0][:3], train_data[1][:3], train_data[2][:3]):
    print('前提:', x0)
    print('假设:', x1)
    print('标签:', y)

训练集约有550000对,测试集约有10000对。下面显示了训练集和测试集中的三个标签"蕴涵""矛盾"和"中性"是平衡的。

In [ ]:
test_data = read_snli(data_dir, is_train=False)
for data in [train_data, test_data]:
    print([[row for row in data[2]].count(i) for i in range(3)])

### 定义用于加载数据集的类

下面我们来定义一个用于加载SNLI数据集的类。类构造函数中的变量`num_steps`指定文本序列的长度,使得每个小批量序列将具有相同的形状。

换句话说,在较长序列中的前`num_steps`个标记之后的标记被截断,而特殊标记"&lt;pad&gt;"将被附加到较短的序列后,直到它们的长度变为`num_steps`。通过实现`__getitem__`功能,我们可以任意访问带有索引`idx`的前提、假设和标签。

In [ ]:
#@save
class SNLIDataset(torch.utils.data.Dataset):
    """用于加载SNLI数据集的自定义数据集"""
    def __init__(self, dataset, num_steps, vocab=None):
        self.num_steps = num_steps
        all_premise_tokens = d2l.tokenize(dataset[0])
        all_hypothesis_tokens = d2l.tokenize(dataset[1])
        if vocab is None:
            self.vocab = d2l.Vocab(all_premise_tokens + \
                all_hypothesis_tokens, min_freq=5, reserved_tokens=['<pad>'])
        else:
            self.vocab = vocab
        self.premises = self._pad(all_premise_tokens)
        self.hypotheses = self._pad(all_hypothesis_tokens)
        self.labels = torch.tensor(dataset[2])
        print('read ' + str(len(self.premises)) + ' examples')

    def _pad(self, lines):
        return torch.tensor([d2l.truncate_pad(
            self.vocab[line], self.num_steps, self.vocab['<pad>'])
                         for line in lines])

    def __getitem__(self, idx):
        return (self.premises[idx], self.hypotheses[idx]), self.labels[idx]

    def __len__(self):
        return len(self.premises)

### 整合代码

现在,我们可以调用`read_snli`函数和`SNLIDataset`类来下载SNLI数据集,并返回训练集和测试集的`DataLoader`实例,以及训练集的词表。

值得注意的是,我们必须使用从训练集构造的词表作为测试集的词表。因此,在训练集中训练的模型将不知道来自测试集的任何新词元。

In [ ]:
#@save
def load_data_snli(batch_size, num_steps=50):
    """下载SNLI数据集并返回数据迭代器和词表"""
    num_workers = d2l.get_dataloader_workers()
    data_dir = d2l.download_extract('SNLI')
    train_data = read_snli(data_dir, True)
    test_data = read_snli(data_dir, False)
    train_set = SNLIDataset(train_data, num_steps)
    test_set = SNLIDataset(test_data, num_steps, train_set.vocab)
    train_iter = torch.utils.data.DataLoader(train_set, batch_size,
                                             shuffle=True,
                                             num_workers=num_workers)
    test_iter = torch.utils.data.DataLoader(test_set, batch_size,
                                            shuffle=False,
                                            num_workers=num_workers)
    return train_iter, test_iter, train_set.vocab

在这里,我们将批量大小设置为128,将序列长度设置为50,并调用`load_data_snli`函数来获取数据迭代器和词表。然后我们打印词表大小。

In [ ]:
train_iter, test_iter, vocab = load_data_snli(128, 50)
len(vocab)

现在我们打印第一个小批量的形状。与情感分析相反,我们有分别代表前提和假设的两个输入`X[0]`和`X[1]`。

In [ ]:
for X, Y in train_iter:
    print(X[0].shape)
    print(X[1].shape)
    print(Y.shape)
    break

## 2.3 可分解注意力模型

### 模型简介

我们介绍了自然语言推断任务和SNLI数据集。鉴于许多模型都是基于复杂而深度的架构,Parikh等人提出用注意力机制解决自然语言推断问题,并称之为"可分解注意力模型"。这使得模型没有循环层或卷积层,在SNLI数据集上以更少的参数实现了当时的最佳结果。

![将预训练GloVe送入基于注意力和MLP的自然语言推断架构](https://zh.d2l.ai/_images/nlp-map-nli-attention.svg)

### 模型架构

与保留前提和假设中词元的顺序相比,我们可以将一个文本序列中的词元与另一个文本序列中的每个词元对齐,然后比较和聚合这些信息,以预测前提和假设之间的逻辑关系。

与机器翻译中源句和目标句之间的词元对齐类似,前提和假设之间的词元对齐可以通过注意力机制灵活地完成。

![利用注意力机制进行自然语言推断](https://zh.d2l.ai/_images/nli-attention.svg)

从高层次上讲,它由三个联合训练的步骤组成:**注意**、**比较**和**汇总**。我们将在下面一步一步地对它们进行说明。

### 步骤1:注意(Attending)

第一步是将一个文本序列中的词元与另一个序列中的每个词元对齐。

假设前提是"我确实需要睡眠",假设是"我累了"。由于语义上的相似性,我们不妨:
- 将假设中的"我"与前提中的"我"对齐
- 将假设中的"累"与前提中的"睡眠"对齐

同样,我们可能希望:
- 将前提中的"我"与假设中的"我"对齐
- 将前提中的"需要"和"睡眠"与假设中的"累"对齐

请注意,这种对齐是使用加权平均的"软"对齐,其中理想情况下较大的权重与要对齐的词元相关联。

#### 注意力权重计算

用$\mathbf{A} = (\mathbf{a}_1, \ldots, \mathbf{a}_m)$和$\mathbf{B} = (\mathbf{b}_1, \ldots, \mathbf{b}_n)$表示前提和假设,其词元数量分别为$m$和$n$,其中$\mathbf{a}_i, \mathbf{b}_j \in \mathbb{R}^{d}$是$d$维的词向量。

对于软对齐,我们将注意力权重$e_{ij} \in \mathbb{R}$计算为:

$$e_{ij} = f(\mathbf{a}_i)^\top f(\mathbf{b}_j)$$

其中函数$f$是多层感知机。

In [ ]:
def mlp(num_inputs, num_hiddens, flatten):
    net = []
    net.append(nn.Dropout(0.2))
    net.append(nn.Linear(num_inputs, num_hiddens))
    net.append(nn.ReLU())
    if flatten:
        net.append(nn.Flatten(start_dim=1))
    net.append(nn.Dropout(0.2))
    net.append(nn.Linear(num_hiddens, num_hiddens))
    net.append(nn.ReLU())
    if flatten:
        net.append(nn.Flatten(start_dim=1))
    return nn.Sequential(*net)

值得注意的是,$f$分别输入$\mathbf{a}_i$和$\mathbf{b}_j$,而不是将它们一对放在一起作为输入。这种**分解**技巧导致$f$只有$m + n$个次计算(线性复杂度),而不是$mn$次计算(二次复杂度)。

对注意力权重进行规范化,我们计算假设中所有词元向量的加权平均值,以获得假设的表示,该假设与前提中索引$i$的词元进行软对齐:

$$\boldsymbol{\beta}_i = \sum_{j=1}^{n}\frac{\exp(e_{ij})}{\sum_{k=1}^{n} \exp(e_{ik})} \mathbf{b}_j$$

同样,我们计算假设中索引为$j$的每个词元与前提词元的软对齐:

$$\boldsymbol{\alpha}_j = \sum_{i=1}^{m}\frac{\exp(e_{ij})}{\sum_{k=1}^{m} \exp(e_{kj})} \mathbf{a}_i$$

In [ ]:
class Attend(nn.Module):
    def __init__(self, num_inputs, num_hiddens, **kwargs):
        super(Attend, self).__init__(**kwargs)
        self.f = mlp(num_inputs, num_hiddens, flatten=False)

    def forward(self, A, B):
        # A/B的形状:(批量大小,序列A/B的词元数,embed_size)
        # f_A/f_B的形状:(批量大小,序列A/B的词元数,num_hiddens)
        f_A = self.f(A)
        f_B = self.f(B)
        # e的形状:(批量大小,序列A的词元数,序列B的词元数)
        e = torch.bmm(f_A, f_B.permute(0, 2, 1))
        # beta的形状:(批量大小,序列A的词元数,embed_size),
        # 意味着序列B被软对齐到序列A的每个词元(beta的第1个维度)
        beta = torch.bmm(F.softmax(e, dim=-1), B)
        # alpha的形状:(批量大小,序列B的词元数,embed_size),
        # 意味着序列A被软对齐到序列B的每个词元(alpha的第1个维度)
        alpha = torch.bmm(F.softmax(e.permute(0, 2, 1), dim=-1), A)
        return beta, alpha

### 步骤2:比较(Comparing)

在下一步中,我们将一个序列中的词元与与该词元软对齐的另一个序列进行比较。请注意,在软对齐中,一个序列中的所有词元(尽管可能具有不同的注意力权重)将与另一个序列中的词元进行比较。

在比较步骤中,我们将来自一个序列的词元的连结(运算符$[\cdot, \cdot]$)和来自另一序列的对齐的词元送入函数$g$(一个多层感知机):

$$\mathbf{v}_{A,i} = g([\mathbf{a}_i, \boldsymbol{\beta}_i]), i = 1, \ldots, m$$
$$\mathbf{v}_{B,j} = g([\mathbf{b}_j, \boldsymbol{\alpha}_j]), j = 1, \ldots, n$$

其中:
- $\mathbf{v}_{A,i}$是指,所有假设中的词元与前提中词元$i$软对齐,再与词元$i$的比较
- $\mathbf{v}_{B,j}$是指,所有前提中的词元与假设中词元$j$软对齐,再与词元$j$的比较

In [ ]:
class Compare(nn.Module):
    def __init__(self, num_inputs, num_hiddens, **kwargs):
        super(Compare, self).__init__(**kwargs)
        self.g = mlp(num_inputs, num_hiddens, flatten=False)

    def forward(self, A, B, beta, alpha):
        V_A = self.g(torch.cat([A, beta], dim=2))
        V_B = self.g(torch.cat([B, alpha], dim=2))
        return V_A, V_B

### 步骤3:聚合(Aggregating)

现在我们有两组比较向量$\mathbf{v}_{A,i}$(i = 1, \ldots, m)和$\mathbf{v}_{B,j}$(j = 1, \ldots, n)。在最后一步中,我们将聚合这些信息以推断逻辑关系。

我们首先求和这两组比较向量:

$$\mathbf{v}_A = \sum_{i=1}^{m} \mathbf{v}_{A,i}, \quad \mathbf{v}_B = \sum_{j=1}^{n}\mathbf{v}_{B,j}$$

接下来,我们将两个求和结果的连结提供给函数$h$(一个多层感知机),以获得逻辑关系的分类结果:

$$\hat{\mathbf{y}} = h([\mathbf{v}_A, \mathbf{v}_B])$$

In [ ]:
class Aggregate(nn.Module):
    def __init__(self, num_inputs, num_hiddens, num_outputs, **kwargs):
        super(Aggregate, self).__init__(**kwargs)
        self.h = mlp(num_inputs, num_hiddens, flatten=True)
        self.linear = nn.Linear(num_hiddens, num_outputs)

    def forward(self, V_A, V_B):
        # 对两组比较向量分别求和
        V_A = V_A.sum(dim=1)
        V_B = V_B.sum(dim=1)
        # 将两个求和结果的连结送到多层感知机中
        Y_hat = self.linear(self.h(torch.cat([V_A, V_B], dim=1)))
        return Y_hat

### 整合代码

通过将注意步骤、比较步骤和聚合步骤组合在一起,我们定义了可分解注意力模型来联合训练这三个步骤。

In [ ]:
class DecomposableAttention(nn.Module):
    def __init__(self, vocab, embed_size, num_hiddens, num_inputs_attend=100,
                 num_inputs_compare=200, num_inputs_agg=400, **kwargs):
        super(DecomposableAttention, self).__init__(**kwargs)
        self.embedding = nn.Embedding(len(vocab), embed_size)
        self.attend = Attend(num_inputs_attend, num_hiddens)
        self.compare = Compare(num_inputs_compare, num_hiddens)
        # 有3种可能的输出:蕴涵、矛盾和中性
        self.aggregate = Aggregate(num_inputs_agg, num_hiddens, num_outputs=3)

    def forward(self, X):
        premises, hypotheses = X
        A = self.embedding(premises)
        B = self.embedding(hypotheses)
        beta, alpha = self.attend(A, B)
        V_A, V_B = self.compare(A, B, beta, alpha)
        Y_hat = self.aggregate(V_A, V_B)
        return Y_hat

## 2.4 训练和评估模型

现在,我们将在SNLI数据集上对定义好的可分解注意力模型进行训练和评估。我们从读取数据集开始。

### 读取数据集

我们使用之前定义的函数下载并读取SNLI数据集。批量大小和序列长度分别设置为256和50。

In [ ]:
batch_size, num_steps = 256, 50
train_iter, test_iter, vocab = d2l.load_data_snli(batch_size, num_steps)

### 创建模型

我们使用预训练好的100维GloVe嵌入来表示输入词元。我们将向量$\mathbf{a}_i$和$\mathbf{b}_j$的维数预定义为100。函数$f$和函数$g$的输出维度被设置为200。然后我们创建一个模型实例,初始化它的参数,并加载GloVe嵌入来初始化输入词元的向量。

In [ ]:
embed_size, num_hiddens, devices = 100, 200, d2l.try_all_gpus()
net = DecomposableAttention(vocab, embed_size, num_hiddens)
glove_embedding = d2l.TokenEmbedding('glove.6b.100d')
embeds = glove_embedding[vocab.idx_to_token]
net.embedding.weight.data.copy_(embeds);

### 训练和评估模型

现在我们可以在SNLI数据集上训练和评估模型。

In [ ]:
lr, num_epochs = 0.001, 4
trainer = torch.optim.Adam(net.parameters(), lr=lr)
loss = nn.CrossEntropyLoss(reduction="none")
d2l.train_ch13(net, train_iter, test_iter, loss, trainer, num_epochs,
    devices)

### 使用模型

最后,定义预测函数,输出一对前提和假设之间的逻辑关系。

In [ ]:
#@save
def predict_snli(net, vocab, premise, hypothesis):
    """预测前提和假设之间的逻辑关系"""
    net.eval()
    premise = torch.tensor(vocab[premise], device=d2l.try_gpu())
    hypothesis = torch.tensor(vocab[hypothesis], device=d2l.try_gpu())
    label = torch.argmax(net([premise.reshape((1, -1)),
                           hypothesis.reshape((1, -1))]), dim=1)
    return 'entailment' if label == 0 else 'contradiction' if label == 1 \
            else 'neutral'

我们可以使用训练好的模型来获得对示例句子的自然语言推断结果。

In [ ]:
predict_snli(net, vocab, ['he', 'is', 'good', '.'], ['he', 'is', 'bad', '.'])

## 小结

### 自然语言推理任务

- 自然语言推理研究"假设"是否可以从"前提"推断出来,其中两者都是文本序列
- 在自然语言推理中,前提和假设之间的关系包括蕴涵关系、矛盾关系和中性关系
- 斯坦福自然语言推理(SNLI)语料库是一个比较流行的自然语言推断基准数据集

### 可分解注意力模型

- 可分解注意模型包括三个步骤来预测前提和假设之间的逻辑关系:**注意、比较和聚合**
- 通过注意力机制,我们可以将一个文本序列中的词元与另一个文本序列中的每个词元对齐,反之亦然。这种对齐是使用加权平均的软对齐,其中理想情况下较大的权重与要对齐的词元相关联
- 在计算注意力权重时,**分解技巧**会带来比二次复杂度更理想的**线性复杂度**
- 我们可以使用预训练好的词向量作为下游自然语言处理任务的输入表示

### 模型优势

1. **简单高效**:不需要循环层或卷积层
2. **可解释性强**:注意力权重可以可视化,显示哪些词元被对齐
3. **计算复杂度低**:分解技巧使得复杂度从$O(mn)$降到$O(m+n)$
4. **效果好**:在SNLI数据集上以更少的参数实现了良好的结果

## 练习

1. 机器翻译长期以来一直是基于翻译输出和翻译真实值之间的表面$n$元语法匹配来进行评估的。可以设计一种用自然语言推断来评价机器翻译结果的方法吗?

2. 我们如何更改超参数以减小词表大小?

3. 使用其他超参数组合训练模型,能在测试集上获得更高的准确度吗?

4. 自然语言推断的可分解注意模型的主要缺点是什么?

5. 假设我们想要获得任何一对句子的语义相似级别(例如,0～1之间的连续值)。我们应该如何收集和标注数据集?请尝试设计一个有注意力机制的模型。